In [42]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from segments_func import *
from sklearn.metrics import accuracy_score,  confusion_matrix
import pandas as pd
from typing import List
from pathlib import Path
from sklearn.metrics import classification_report, precision_recall_fscore_support


In [43]:
from typing import Optional

labels = ["HSIL", "LSIL", "NSIL"]

NSIL_center = 6.028023671609743
NSIL_half   = 8.003730131911812
LSIL_center = 17.960424045561066
LSIL_half   = 11.085963449966334
HSIL_center = 35.708751454451765
HSIL_half   = 10.645653397929129

NSIL_range = (float(NSIL_center - NSIL_half), float(NSIL_center + NSIL_half))
LSIL_range = (float(LSIL_center - LSIL_half), float(LSIL_center + LSIL_half))
HSIL_range = (float(HSIL_center - HSIL_half), float(HSIL_center + HSIL_half))
CENTERS = {"NSIL": NSIL_center, "LSIL": LSIL_center, "HSIL": HSIL_center}

def iter_images_with_labels(root: Path):
    for label in ["HSIL", "LSIL", "NSIL"]:
        class_dir = root / label
        if not class_dir.exists():
            continue
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                yield str(p), label

def classify_by_ratio(ratio: float) -> Optional[str]:
    """
    - jeśli ratio wpada do kilku przedziałów -> wybierz klasę o najbliższym środku,
    - jeśli ratio poza wszystkimi -> najbliższy środek,
    - jeśli ratio to NaN -> zwróć None (próbka będzie pominięta).
    """
    if ratio is None or (isinstance(ratio, float) and np.isnan(ratio)):
        return None

    covered = []
    if NSIL_range[0] <= ratio <= NSIL_range[1]:
        covered.append("NSIL")
    if LSIL_range[0] <= ratio <= LSIL_range[1]:
        covered.append("LSIL")
    if HSIL_range[0] <= ratio <= HSIL_range[1]:
        covered.append("HSIL")

    if len(covered) == 1:
        return covered[0]
    elif len(covered) > 1:
        return min(covered, key=lambda lab: abs(ratio - CENTERS[lab]))
    else:
        return min(CENTERS.keys(), key=lambda lab: abs(ratio - CENTERS[lab]))

def main(test_folder, name_output_dir):
    y_true: List[str] = []
    y_pred: List[str] = []
    records: List[dict] = []

    for img_path, true_label in iter_images_with_labels(test_folder):
        ratio = get_mean_ratio_file(img_path)
        pred_label = classify_by_ratio(ratio)

        if pred_label is None:
            print(f"[WARN] Pomijam: {img_path} (ratio={ratio})")
            continue

        y_true.append(true_label)
        y_pred.append(pred_label)
        records.append({
            "image_path": img_path,
            "true_label": true_label,
            "pred_label": pred_label,
            "mean_ratio": ratio,
        })

    if not y_true:
        raise SystemExit("Brak danych do ewaluacji – upewnij się, że katalog 'test' zawiera podfoldery HSIL/LSIL/NSIL z obrazami.")

    acc = accuracy_score(y_true, y_pred)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="weighted", zero_division=0
    )

    print("\n=== METRYKI (global) ===")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Precision (macro):   {prec_macro:.4f}")
    print(f"Recall    (macro):   {rec_macro:.4f}")
    print(f"F1        (macro):   {f1_macro:.4f}")
    print(f"Precision (weighted):{prec_weighted:.4f}")
    print(f"Recall    (weighted):{rec_weighted:.4f}")
    print(f"F1        (weighted):{f1_weighted:.4f}")

    print("\n=== RAPORT (per klasa) ===")
    print(classification_report(y_true, y_pred, labels=labels, digits=4))

    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize='true')
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in labels],
        columns=[f"pred_{l}" for l in labels],
    )
    print("\n=== CONFUSION MATRIX (znormalizowana, wartości 0–1) ===")
    print(cm_df)

    out_dir = Path(name_output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(records).to_csv(out_dir / "predictions.csv", index=False)
    cm_df.to_csv(out_dir / "confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title("Confusion Matrix (HSIL/LSIL/NSIL, normalized)")
    tick_marks = np.arange(len(labels))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="black")

    ax.set_ylabel("True class")
    ax.set_xlabel("Predicted class")
    fig.tight_layout()
    plt.savefig(out_dir / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.close()


In [44]:
main(Path(r'C:\Users\aleks\OneDrive\Documents\inzynierka\data\data_single_cropped3\test'), name_output_dir="evaluation_out")


=== METRYKI (global) ===
Accuracy:            0.7864
Precision (macro):   0.7821
Recall    (macro):   0.7840
F1        (macro):   0.7810
Precision (weighted):0.7910
Recall    (weighted):0.7864
F1        (weighted):0.7865

=== RAPORT (per klasa) ===
              precision    recall  f1-score   support

        HSIL     0.8571    0.7500    0.8000        40
        LSIL     0.7143    0.7407    0.7273        27
        NSIL     0.7750    0.8611    0.8158        36

    accuracy                         0.7864       103
   macro avg     0.7821    0.7840    0.7810       103
weighted avg     0.7910    0.7864    0.7865       103


=== CONFUSION MATRIX (znormalizowana, wartości 0–1) ===
           pred_HSIL  pred_LSIL  pred_NSIL
true_HSIL   0.750000   0.150000   0.100000
true_LSIL   0.074074   0.740741   0.185185
true_NSIL   0.083333   0.055556   0.861111


In [45]:
# zbiór testowy wyciętych zdjęć 
main(Path(r'C:\Users\aleks\OneDrive\Documents\inzynierka\data\TEST'), name_output_dir="evaluation_out_test_mine")


=== METRYKI (global) ===
Accuracy:            0.7000
Precision (macro):   0.6804
Recall    (macro):   0.7000
F1        (macro):   0.6821
Precision (weighted):0.6804
Recall    (weighted):0.7000
F1        (weighted):0.6821

=== RAPORT (per klasa) ===
              precision    recall  f1-score   support

        HSIL     0.6364    0.7000    0.6667        10
        LSIL     0.5714    0.4000    0.4706        10
        NSIL     0.8333    1.0000    0.9091        10

    accuracy                         0.7000        30
   macro avg     0.6804    0.7000    0.6821        30
weighted avg     0.6804    0.7000    0.6821        30


=== CONFUSION MATRIX (znormalizowana, wartości 0–1) ===
           pred_HSIL  pred_LSIL  pred_NSIL
true_HSIL        0.7        0.3        0.0
true_LSIL        0.4        0.4        0.2
true_NSIL        0.0        0.0        1.0

Accuracy:            0.7000
Precision (macro):   0.6804
Recall    (macro):   0.7000
F1        (macro):   0.6821
Precision (weighted):0.6804